## Preparacion del sistema

In [1]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
import os

llm= ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    temperature=0,
    streaming=True
)

print("Configuracion exitosa")

Configuracion exitosa


## Base de conocimiento y historial del chat


In [2]:
historial_chat = []

Base_conocimientos={
    "ayuno": 
    """Se recomienda ayuno de 6 a 8 horas antes de una cirugía.""",

    "medicamentos": 
    """Debe consultar con su médico antes de suspender cualquier medicamento.""",

    "alergias": 
    """Es importante informar al equipo médico sobre cualquier alergia.""",

    "higiene": 
    """Se recomienda bañarse antes del procedimiento según indicaciones médicas.""",

    "alcohol": 
    """Evitar consumo de alcohol al menos 24 horas antes de la cirugía.""",

    "tiempo": 
    """Se recomienda llegar con aticipasion al centro medico"""
}

#Busqueda del contexto

def buscar_contexto(pregunta):
    contexto = []
    pregunta = pregunta.lower()

    for clave, valor in Base_conocimientos.items():
        if clave in pregunta:
            contexto.append(valor)

        if not contexto:
            contexto.append("""Siga siempre las indicaciones generales de su equipo medico""")
        return contexto
    
print("Base de conocimiento lista y recuperacion de contexto")

Base de conocimiento lista y recuperacion de contexto


## Planificador y herramientas

In [3]:
def planificar_tarea(consulta):
    consulta = consulta.lower()

    if "reporte" in consulta:
        return [
            "Buscar informacion",
            "Generar reporte",
            "Entregar resultado"
        ]

    elif "riesgo" in consulta:
        return [
            "Analizar antecedentes",
            "Evaluar riesgo",
            "Entregar evaluacion"
        ]

    elif "recordatorio" in consulta:
        return [
            "Registrar recordatorio",
            "Generar respuesta"
        ]
    return [
        "Buscar recomendaciones",
        "Generar respuesta"
    ]

@tool
def buscar_recomendacion(pregunta:str) -> str:
    """
    Busca recomendaciones medicas preoperatorias
    """

    docs = buscar_contexto(pregunta)

    return "\n".join(docs)

@tool
def generar_reporte(texto:str)-> str:
    """
    Genera un reporte resmido
    """

    return f"""
    REPORTE PREOPERATORIO
    {texto}
    """

@tool
def evaluar_riesgo(texto:str)-> str:
    """
    Evaluá riesgos simples.
    """

    texto = texto.lower()

    if "alcohol" in texto:
        return "Posible incumplimiento de indicaciones preoperatorias"
    
    if "alergia" in texto:
        return "Se detectan anteedentes de alergias"
    
    return "No se identifican riesgos relevantes."

@tool
def registro_recordatorio(texto:str):
    """
    Registra recordatorios para el paciente
    """

    return f"recordatorio registrado {texto}"

print("Herramientas y planificacion agregados")


Herramientas y planificacion agregados


## Agente

In [4]:
agent = create_agent(
    model=llm,
    tools = [
        buscar_recomendacion,
        generar_reporte,
        evaluar_riesgo,
        registro_recordatorio
    ],
    system_prompt="""
        Eres un asistente de una clinica el cual esta especilizado en recomendaciones preoperatorias. 
        Tienes que seguir el siguiente reglamento:
            -Solo dar recomendaciones generales.
            -NO puedes recomendar medicamentos ni dosis.
            -NO realizar diagnosticos.
            -Si te piden realizar un diagnostico debes rechazarlo y explicar el pq no debes.
        
        Tu funcion principal es orientar a los pacientes antes de una cirugia utilizando las herramientas
        disponibles cuando sean necesarias y Nunca inventes información médica.
    """
)

print("agente implementado")

agente implementado


## Codigo general

In [10]:

def ChatBot_Medico2():
    print("==ChatBot preoperatorio==")
    print("Escribe 'salir' para terminar la conversación\\n")

    while True:
        pregunta = input("\nUsuario:")

        if pregunta.lower()=="salir":
            print("Gracias por utilizar: ChatBot clinica titirilquen")
            break

        documentos = buscar_contexto(pregunta)

        contexto = "\n".join(documentos)

        plan = planificar_tarea(pregunta)

        plan_texto = "\n".join(plan)

        mensaje_usuario = f"""
        Plan de acción:

        {plan_texto}

        Contexto recuperado:{contexto}

        Consulta del paciente: {pregunta}
        """

        historial_chat.append(
            {
                "role": "user",
                "content": mensaje_usuario
            }
        )

        try:
            resultado = agent.invoke(
                {
                    "messages":historial_chat
                }
            )

            respuesta = resultado["messages"][-1].content

            print("\nAsistente: ")
            print(respuesta)

            historial_chat.append(
                {
                    "role": "assistant",
                    "content": respuesta
                }
            )

        except Exception as e:
            print(f"\nError: {e}")

ChatBot_Medico2()  

==ChatBot preoperatorio==
Escribe 'salir' para terminar la conversación\n

Asistente: 
Antes de una cirugía de apéndice, es importante seguir siempre las indicaciones generales de su equipo médico. Algunas recomendaciones generales incluyen:

- Mantener el ayuno según lo indicado por su médico (no comer ni beber durante varias horas antes de la cirugía).
- Informar a su equipo médico sobre cualquier medicamento o suplemento que esté tomando.
- Mantener una buena higiene personal, como bañarse antes del procedimiento.
- Llegar a tiempo al hospital o clínica.
- Llevar consigo documentos importantes y resultados de estudios previos.

Recuerde que su equipo médico le dará instrucciones específicas para su caso. Si tiene alguna duda, consulte directamente con ellos.

Asistente: 
He registrado un recordatorio para que se le recuerde la fecha y hora de su cirugía. Si necesita que se le envíe un recordatorio específico, por favor indíqueme la fecha y hora exacta, y lo agregaré para usted.

Asi